# 📘 Module 12.6 — AdamW

## 🔥 Core Idea

**AdamW = Adam + decoupled weight decay**

Adam already gives us:

* Momentum-like behavior through the **first moment** $m_t$
* RMSProp-like adaptive scaling through the **second moment** $v_t$
* Bias correction

AdamW keeps all of that.

The important change is **how weight decay is applied**.

> **Adam:** regularization is coupled into the gradient that Adam processes.
> **AdamW:** Adam handles the loss gradient, while weight decay directly shrinks the weights.

This distinction was introduced in the **Decoupled Weight Decay Regularization** work by Loshchilov & Hutter. ([arXiv][1])

![Image](https://images.openai.com/static-rsc-4/FU-2fR_NO3tJWfXPk8Hqsur3BK-D-3rw17nBNdStWuHFgZ3-4y8G84BK5axtXFg6_3GBG0XdMC1OD4UaIrpqvk_j2d6SCOHBvxHM76sHl8B7G5Ex5-dqHVxocE94GHmSGiNswH5H_EpJWbRz56gQOVnkex00BcecRbSaDIndgQeQ5ovNeQu9fo4Elc44F81w?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/oDlulD9TdaFKHhYMPQ3wL-8hOT5NwSrx7ckFHPCI63qsGuBADqwqgdpgpxW3zvLgviMmkj-vwgDr72ONTGnEeOCLed0QV1JXZhJpcZqgxlVxXYaPtBJktT99EgbentA4vzgcDC2SAIsrFyg3OMeHPeqgbsox4wFHsSBifp1NF3zEKaF925druAeSE9RhgO-4?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/dEMpOqN9xy0TpadOwql_Ag9qhgJZ95zXus-CuAO4slN7TeqDPMvsI4ukBG30S_oQ6xUTEfA919GHbKPBowpvXw7T0INyb3qnUGHKSW5lOcscNjVIeov_qpwCCaP-aVEJ2KVHikkcTPsBa06MS5y_RfOpK5ISF-iPjAuOLsbkvm6a3B9bRCZnvZ1Dawruc0DR?purpose=fullsize)

---

# 1. 🔥 Why Do We Need AdamW?

First remember what we learned in regularization.

We want the model to avoid unnecessarily large weights.

For L2 regularization:

$$
J(\theta)=L(\theta)+\frac{\lambda}{2}\|\theta\|^2
$$

Taking the gradient:

$$
\nabla_\theta J
=
\nabla_\theta L
+
\lambda\theta
$$

So the regularization term becomes part of the gradient.

With ordinary **SGD**, this gives:

$$
\theta_t
=
\theta_{t-1}
-
\eta(\nabla L+\lambda\theta)
$$

Rearrange:

$$
\theta_t
=
(1-\eta\lambda)\theta_{t-1}
-
\eta\nabla L
$$

Notice something important:

### The weights are directly shrinking.

That is **weight decay**.

For ordinary SGD, L2 regularization and weight decay are mathematically equivalent under the usual formulation. ([arXiv][1])

---

# 2. 🔥 The Problem With Adam

Adam doesn't simply do:

$$
\theta \leftarrow \theta-\eta g
$$

Instead, Adam modifies the gradient using its adaptive moments.

Recall:

### First moment

$$
m_t
=
\beta_1m_{t-1}
+
(1-\beta_1)g_t
$$

### Second moment

$$
v_t
=
\beta_2v_{t-1}
+
(1-\beta_2)g_t^2
$$

### Bias correction

$$
\hat m_t
=
\frac{m_t}{1-\beta_1^t}
$$

$$
\hat v_t
=
\frac{v_t}{1-\beta_2^t}
$$

### Adam update

$$
\theta_t
=
\theta_{t-1}
-
\eta
\frac{\hat m_t}
{\sqrt{\hat v_t}+\epsilon}
$$

Now imagine we add L2 regularization:

$$
g_t
=
\nabla L+\lambda\theta
$$

Adam doesn't know:

> "This part is the actual loss gradient and this part is regularization."

It processes the **combined gradient** through its adaptive machinery.

That's the problem.

---

# 3. 🔥 What Does "Decoupled" Mean?

This is the **most important concept in AdamW**.

### Adam with L2-style regularization

Conceptually:

$$
\boxed{
\text{loss gradient}
+
\text{regularization}
\rightarrow
\text{Adam}
}
$$

The regularization term gets passed through Adam's adaptive scaling.

### AdamW

Instead:

$$
\boxed{
\text{loss gradient}
\rightarrow
\text{Adam}
}
$$

and separately:

$$
\boxed{
\text{weights}
\rightarrow
\text{direct shrinkage}
}
$$

So they are **decoupled**.

![Image](https://images.openai.com/static-rsc-4/FU-2fR_NO3tJWfXPk8Hqsur3BK-D-3rw17nBNdStWuHFgZ3-4y8G84BK5axtXFg6_3GBG0XdMC1OD4UaIrpqvk_j2d6SCOHBvxHM76sHl8B7G5Ex5-dqHVxocE94GHmSGiNswH5H_EpJWbRz56gQOVnkex00BcecRbSaDIndgQeQ5ovNeQu9fo4Elc44F81w?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/onSVL3pwWb8tt74oogNUxNsuxfEt6bTCx0OYalukPqmP0UFv-msHLSmWISCi_I_ka4C7QoEYhLWMJGHJEHyrlDL0ZBSzifSWfWqcbOefWIfRaz0GMvN6o2B0B_2U8uPrC81WY4_RmjEltPsROzDELLj_4TXasNJdUMu1VwOhr2KM8DQJmCOXUZuvJiDqh-9T?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/dEMpOqN9xy0TpadOwql_Ag9qhgJZ95zXus-CuAO4slN7TeqDPMvsI4ukBG30S_oQ6xUTEfA919GHbKPBowpvXw7T0INyb3qnUGHKSW5lOcscNjVIeov_qpwCCaP-aVEJ2KVHikkcTPsBa06MS5y_RfOpK5ISF-iPjAuOLsbkvm6a3B9bRCZnvZ1Dawruc0DR?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/J5a6u1eDi4GLwkBeKH5LzY5wVS4-wGvMXEbcRx1UXJL81SKNmVtLw22I9kL6m2MWIFaCdB0Gl8KlM_uiy5JKzFNziWP9hroVjFmbiXZItEvEqmXu8hHDBHIvPj1ZCMzdhxvcHo9BJ4Lygkuu9EUUiWgBLG6EHn_T9pQLhl1QMkpYxUvKk14014qFAY2IjZef?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/Rvt2TZlEiXId1I-oRst9Vb4rq92MmfQ387zk0Vo_RkrBZutOOAmZPtaOaPDjUZFdF_tfi7ojyGWBL-KvJX3PewIsr6lmcpfXsSIy4w2HNR2MK8JWhFdm9FSiYhcCJZsd2ppx1TEoboaW73gbmWcXnWELqD3BLxuNE4M9axtUDUwz4y2AuUyhZNC-euEDxZfZ?purpose=fullsize)

---

# 4. 🔥 AdamW Mathematics

AdamW still calculates the same moments.

### First moment

$$
m_t
=
\beta_1m_{t-1}
+
(1-\beta_1)g_t
$$

### Second moment

$$
v_t
=
\beta_2v_{t-1}
+
(1-\beta_2)g_t^2
$$

### Bias correction

$$
\hat m_t
=
\frac{m_t}{1-\beta_1^t}
$$

$$
\hat v_t
=
\frac{v_t}{1-\beta_2^t}
$$

Then AdamW applies the weight decay **directly to the parameter**.

A common compact form is:

$$
\boxed{
\theta_t
=
(1-\eta\lambda)\theta_{t-1}
-
\eta
\frac{\hat m_t}
{\sqrt{\hat v_t}+\epsilon}
}
$$

Think of it as two separate actions:

### ① Adam update

$$
-\eta
\frac{\hat m_t}
{\sqrt{\hat v_t}+\epsilon}
$$

### ② Weight decay

$$
-\eta\lambda\theta_{t-1}
$$

Therefore:

$$
\boxed{
\text{AdamW}
=
\text{Adam optimization}
+
\text{direct weight shrinkage}
}
$$

---

# 5. 🧠 Simple Intuition

Imagine your weight is:

$$
w=10
$$

and weight decay wants to shrink it.

Instead of saying:

> "Let's modify the gradient and then let Adam decide how strongly this regularization acts."

AdamW says:

> "Adam, you handle the actual learning gradient."

> "Weight decay, you independently shrink the weight."

So:

```text
                LOSS
                  ↓
            Loss Gradient
                  ↓
          ┌───────────────┐
          │     ADAM      │
          │ m, v, bias    │
          │ correction     │
          └───────┬───────┘
                  ↓
             Adam update
                  │
                  │
                  ↓
        ┌──────────────────┐
        │ Parameter update │
        │                  │
        │ + direct decay   │
        └──────────────────┘
                  ↓
             New weights
```

That's the whole idea of **decoupled weight decay**.

---

# 6. 🔥 Adam vs AdamW

| Feature                 | Adam                                    | AdamW                                  |
| ----------------------- | --------------------------------------- | -------------------------------------- |
| First moment            | ✅                                       | ✅                                      |
| Second moment           | ✅                                       | ✅                                      |
| Bias correction         | ✅                                       | ✅                                      |
| Adaptive learning rate  | ✅                                       | ✅                                      |
| Weight decay            | Coupled with optimization               | **Decoupled**                          |
| Direct weight shrinkage | Not in the standard coupled formulation | ✅                                      |
| Main idea               | Adaptive optimization                   | Adaptive optimization + separate decay |

### Memory trick 🧠

> **Adam learns how to move.**
> **AdamW learns how to move + independently shrinks the weights.**

---

# 7. 🔥 L2 Regularization vs Weight Decay

This is where your previous doubt becomes important.

You asked earlier whether L2 is used to shrink weights.

### Yes.

L2 regularization:

$$
R(\theta)=\frac12\|\theta\|^2
$$

creates a gradient:

$$
\nabla R(\theta)=\theta
$$

which encourages weights toward zero.

But there is an important distinction:

### With SGD

L2 regularization and weight decay can be expressed equivalently:

$$
\theta_t
=
(1-\eta\lambda)\theta_{t-1}
-
\eta g_t
$$

### With Adam

Because Adam adaptively rescales gradients, putting $\lambda\theta$ inside the gradient means the regularization gets affected by Adam's adaptive mechanism.

Therefore:

> **L2 regularization and weight decay are not generally equivalent for Adam.** ([arXiv][1])

AdamW fixes this by **decoupling** the decay from Adam's gradient transformation.

---

# 8. 🔥 Why Is This Useful?

The original AdamW work argued that decoupling weight decay makes the regularization behavior more independent of the adaptive optimization mechanism and reported improved generalization in their experiments. ([arXiv][1])

Conceptually:

### Adam

```text
Loss gradient
     +
λ × weight
     ↓
   Adam
     ↓
  Update
```

### AdamW

```text
Loss gradient ──→ Adam ──→ Adam update
                              │
                              ↓
                    Direct weight decay
                              │
                              ↓
                         New weights
```

This separation is the key.

---

# 9. 🔥 AdamW in PyTorch

```python
import torch

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.001,
    weight_decay=0.01
)
```

The important hyperparameters:

| Parameter      | Meaning                             |
| -------------- | ----------------------------------- |
| `lr`           | Learning rate                       |
| `weight_decay` | Strength of direct weight shrinkage |
| `betas`        | First/second moment decay rates     |
| `eps`          | Numerical stability                 |

Typical starting values often look like:

```python
lr = 0.001
weight_decay = 0.01
betas = (0.9, 0.999)
```

But these are **starting points, not universal rules**.

---

# 10. ⚠️ Don't Confuse These

### AdamW ≠ Dropout

**Dropout**

Randomly removes activations during training.

```text
Neuron → randomly OFF
```

### AdamW weight decay

Shrinks parameters.

```text
w → slightly smaller
```

So:

| Technique    | Acts on                     |
| ------------ | --------------------------- |
| Dropout      | Activations                 |
| Weight decay | Parameters/weights          |
| BatchNorm    | Activations/statistics      |
| LayerNorm    | Features within each sample |

---

# 11. 🔥 AdamW Does NOT Replace Adam's Moments

Very important.

Some beginners think:

> AdamW is a completely different optimizer.

Not really.

AdamW still has:

$$
m_t
$$

and

$$
v_t
$$

and bias correction.

The major change is **how weight decay is handled**.

So:

```text
Adam
 ├── First moment
 ├── Second moment
 ├── Bias correction
 └── Adaptive update

AdamW
 ├── First moment
 ├── Second moment
 ├── Bias correction
 ├── Adaptive update
 └── + Decoupled weight decay
```

---

# 12. 🧩 Small Numerical Intuition

Suppose:

$$
w=10
$$

Learning rate:

$$
\eta=0.01
$$

Weight decay:

$$
\lambda=0.1
$$

The decay component is:

$$
\eta\lambda w
$$

$$
=0.01\times0.1\times10
$$

$$
=0.01
$$

So the direct decay contribution is:

$$
-0.01
$$

giving:

$$
w\rightarrow9.99
$$

**while Adam simultaneously performs its normal adaptive gradient update.**

The actual new weight depends on the Adam gradient update too.

---

# 13. 🔥 What You Should Remember for Industry

When you see:

```python
torch.optim.AdamW(...)
```

immediately think:

> **Adaptive Adam optimization + decoupled weight decay**

And when you see:

```python
weight_decay=0.01
```

think:

> **Encourage parameters to remain smaller by directly shrinking them during optimization.**

Don't think:

> "Drop some neurons."

That's dropout.

---

# 14. 🎯 Common Mistakes

### ❌ Mistake 1

**"AdamW doesn't use Adam."**

Wrong.

AdamW retains Adam's moment estimates and bias correction.

---

### ❌ Mistake 2

**"Weight decay and L2 are always identical."**

Not for adaptive optimizers like Adam.

The AdamW paper specifically highlights this distinction. ([arXiv][1])

---

### ❌ Mistake 3

**"AdamW removes large weights by clipping them."**

No.

Gradient clipping and weight decay are different.

```text
Gradient clipping → controls gradient magnitude

Weight decay → shrinks parameters
```

---

### ❌ Mistake 4

**"Higher weight decay always improves generalization."**

No.

Too much regularization can prevent the model from learning effectively. The weight-decay coefficient needs to be tuned for the task.

---

# 15. 🎤 Interview / GATE Points

### Q1. What is AdamW?

> AdamW is Adam with **decoupled weight decay**, where parameter shrinkage is applied separately from Adam's adaptive gradient update.

### Q2. Why was AdamW introduced?

> Because L2 regularization and weight decay are not generally equivalent when used with adaptive optimizers such as Adam.

### Q3. What does "decoupled" mean?

> The weight-decay operation is separated from the gradient processed by Adam.

### Q4. Does AdamW still use momentum?

Yes — through the first moment $m_t$.

### Q5. Does AdamW still use the second moment?

Yes — through $v_t$.

### Q6. Main difference?

$$
\boxed{\text{AdamW = Adam + decoupled weight decay}}
$$

---

# 🧠 Final Cheat Sheet

```text
SGD
 ↓
Momentum
 ↓
AdaGrad
 ↓
RMSProp
 ↓
Adam
 ├── First moment → direction/momentum
 ├── Second moment → adaptive scaling
 └── Bias correction
 ↓
AdamW
 └── Adam + decoupled weight decay
```

### Core formulas

**Adam:**

$$
m_t=\beta_1m_{t-1}+(1-\beta_1)g_t
$$

$$
v_t=\beta_2v_{t-1}+(1-\beta_2)g_t^2
$$

$$
\hat m_t=\frac{m_t}{1-\beta_1^t}
$$

$$
\hat v_t=\frac{v_t}{1-\beta_2^t}
$$

**AdamW:**

$$
\boxed{
\theta_t
=
(1-\eta\lambda)\theta_{t-1}
-
\eta
\frac{\hat m_t}
{\sqrt{\hat v_t}+\epsilon}
}
$$

### One-line memory trick 🚀

> **Adam adapts the gradient; AdamW adapts the gradient AND independently decays the weights.**

**Primary reference:** Loshchilov & Hutter, *Decoupled Weight Decay Regularization*. ([arXiv][1])

[1]: https://arxiv.org/abs/1711.05101?utm_source=chatgpt.com "Decoupled Weight Decay Regularization"
---